[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CCS-ZCU/EMLAP_ETL/blob/master/scripts/colab-explorations.ipynb)

In [4]:
import pandas as pd
import requests
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import matplotlib.pyplot as plt

In [5]:
emlap_metadata = pd.read_csv("https://raw.githubusercontent.com/CCS-ZCU/EMLAP_ETL/refs/heads/master/data/emlap_metadata.csv", sep=";")
emlap_metadata.head(5)

,working_title,No.,is_done,is_noscemus,if_noscemus_id,"#if is_noscemus = True, don't transcribe",AUTHORSHIP,is_one_author,#if more than 1 author skip section and choose compendium below,is_author_known,...,link,source_of_file,origin_of_copy,REFERENCES,catalogue_reference,secondary_references,general_comments,OTHER,filename,Unnamed: 64
0,"Augurello, Chrysopoeia",100001,True,True,713324.0,NaN,NaN,True,NaN,True,...,https://wiki.uibk.ac.at/noscemus/Chrysopoeia,Noscemus,Unknown,NaN,Noscemus Wiki,Soranzo 2019,The 1518 Basel version is also in Noscemus,NaN,"Augurello,_Giovanni_Aurelio_-_Chrysopoeia__Ven...",NaN
1,"Pseudo-Lull, Secretis",100002,True,False,NaN,NaN,NaN,True,NaN,True,...,https://www.digitale-sammlungen.de/en/view/bsb...,MDZ,MBS,NaN,Hirsch 1950,NaN,"There is a prior, 1514 edition of De secretis ...",NaN,Pseudo-Lull1518_De_secretis_naturae_MDZ.pdf,NaN
2,"Pantheus, Ars Transmutatione",100003,True,False,NaN,NaN,NaN,True,NaN,True,...,NaN,GB,BL,NaN,NaN,NaN,This book was first published in 1518 with an ...,NaN,Pantheus1518_Ars_Transmutationis_Metallicae_BL...,NaN
3,"Pantheus, Commentarium",100004,True,False,NaN,NaN,NaN,True,NaN,True,...,https://www.digitale-sammlungen.de/en/view/bsb...,MDZ,MSB,NaN,NaN,NaN,This 1519 book is catalogued wrongly by many l...,NaN,Pantheus1519_Commentarium_Transmutationis_Meta...,NaN
4,"Pantheus, Voarchadumia",100005,True,False,NaN,NaN,NaN,True,NaN,True,...,NaN,ONB,ONB,NaN,NaN,NaN,Dedicated to Leonellus Marquis of Estense,NaN,Pantheus1530_Voarchadumia_ONB.pdf,NaN


In [6]:
# load sentence data for a specific file based on its ID:
work_id = 100002
dir = "data/sents_data"
filename = str(work_id) + ".json"
url = "https://raw.githubusercontent.com/CCS-ZCU/EMLAP_ETL/refs/heads/master/{0}/{1}".format(dir,filename)
resp = requests.get(url)
sents_data = resp.json()
# look at a random selection of morphologically annotated and lemmatized sentences
sents_data[0:50]

[[100002,
  0,
  'ctoris Raymundi Lulii de secretis nature siue de quinta essentia De secretis nature libellus.',
  [['ctoris', 'ctor', 'NOUN', [0, 6], [5], [1]],
   ['Raymundi', 'Raymundus', 'PROPN', [7, 15], [5], [1]],
   ['Lulii', 'Lulius', 'PROPN', [16, 21], [5], [1]],
   ['de', 'de', 'ADP', [22, 24], [5], [1]],
   ['secretis', 'secerno', 'VERB', [25, 33], [5], [1]],
   ['nature', 'natura', 'NOUN', [34, 40], [5], [2]],
   ['siue', 'siue', 'CCONJ', [41, 45], [5], [2]],
   ['de', 'de', 'ADP', [46, 48], [5], [2]],
   ['quinta', 'quintus', 'ADJ', [49, 55], [5], [2]],
   ['essentia', 'essentia', 'NOUN', [56, 64], [5], [2]],
   ['De', 'de', 'ADP', [65, 67], [7], [0]],
   ['secretis', 'secerno', 'VERB', [68, 76], [7], [0]],
   ['nature', 'natura', 'NOUN', [77, 83], [7], [0]],
   ['libellus', 'libellus', 'NOUN', [84, 92], [7], [0]],
   ['.', '.', 'PUNCT', [92, 93], [7], [0]]]],
 [100002,
  1,
  'Incipit liber prime distinctionis secretorum nature seu quinte essentie sacri doctoris magistri

On the top level, this object represents the whole document as a list of comprehensive morphological and topological data for each sentence within if.

For each sentence, there is:

    (a) the id ("No.") of the source document within the EMLAP catalogue;
    (b) the positional index of the sentence within the document (following Python convention starting with 0)
    (c) the raw text of the sentence
    (d) morphological and positional data for each token, which consist of:
        d-1) text of the token
        d-2) lemma corresponding to the token
        d-3) part-of-speech of the token
        d-4) start and end positional index of the token within the sentence
        d-5) index of the page (or pages) within the PDF from which the token comes from
        d-6) index of the textblock within the PDF page as rendered by the PyMuPDF library)




In [10]:
# extract from them only lemmata of certain part-of-speech tags
doc_lemmatized_sents = []
for sent_data in sents_data:
    lemmatized_sent = []
    for token_data in sent_data[3]:
        if (token_data[2] in ["NOUN", "VERB", "ADJ", "PROPN"]) & (token_data[1] != "") :
            lemmatized_sent.append(token_data[1])
    doc_lemmatized_sents.append(lemmatized_sent)
doc_lemmatized_sents[:10]

[['Iohannes',
  'Aurelius',
  'Augurellius',
  'P.',
  'Ariminensis',
  'Chrysopoeia',
  'Liber',
  'Iius'],
 ['Liber', 'primus'],
 ['aurifer', 'paruus', 'animus', 'uis', 'Artes', 'quaero'],
 ['longus', 'tempus', 'pario'],
 ['res', 'inuolucrum', 'tantus', 'euoluio', 'moles', 'possum'],
 ['clarus', 'perhibeo', 'carmen', 'Eusimus'],
 ['Musa', 'commendo', 'almus', 'numerus', 'facio', 'prior'],
 ['opus', 'autor'],
 ['nomen', 'tutus', 'pergo'],
 ['opto']]

In [11]:
# this format of data is ideal for various semantic oriented  co-occurrence analyses

# another format of the data is a flat list of lemmata:
doc_lemmata_list = [l for sent in doc_lemmatized_sents for l in sent]
#with this you can immediatelly proceed to calculate word frequencies:
doc_word_freqs = nltk.FreqDist(doc_lemmata_list).most_common()
doc_word_freqs[:50]

[('possum', 91),
 ('uideo', 90),
 ('res', 85),
 ('ars', 74),
 ('aurum', 73),
 ('fero', 69),
 ('facio', 62),
 ('uis', 57),
 ('opus', 49),
 ('tempus', 47),
 ('ignis', 47),
 ('primus', 46),
 ('natura', 45),
 ('dico', 41),
 ('longus', 40),
 ('do', 40),
 ('labor', 39),
 ('uarius', 38),
 ('terra', 38),
 ('mens', 37),
 ('summus', 37),
 ('pars', 36),
 ('duco', 36),
 ('metallum', 35),
 ('quaero', 33),
 ('magnus', 33),
 ('uerus', 33),
 ('seruo', 32),
 ('proprius', 31),
 ('calor', 31),
 ('mortalis', 30),
 ('uita', 30),
 ('pondus', 30),
 ('manus', 29),
 ('mirus', 28),
 ('corpus', 28),
 ('argentum', 28),
 ('color', 28),
 ('caelum', 27),
 ('doceo', 27),
 ('ago', 27),
 ('refero', 27),
 ('unda', 26),
 ('uolo', 25),
 ('puluis', 25),
 ('semen', 25),
 ('alo', 25),
 ('quondam', 25),
 ('paruus', 24),
 ('uirtus', 24)]

In [12]:
# the same way we can load and analyze the whole corpus or any subselection of it based on the IDS:
# let's define a function automating the process:
def extract_lemmatized_sents(work_id):
    dir = "data/sents_data"
    filename = str(work_id) + ".json"
    url = "https://raw.githubusercontent.com/CCS-ZCU/EMLAP_ETL/refs/heads/master/{0}/{1}".format(dir,filename)
    resp = requests.get(url)
    sents_data = resp.json()
    doc_lemmatized_sents = []
    for sent_data in sents_data:
        lemmatized_sent = []
        for token_data in sent_data[3]:
            if (token_data[2] in ["NOUN", "VERB", "ADJ", "PROPN"]) & (token_data[1] != "") :
                lemmatized_sent.append(token_data[1])
        doc_lemmatized_sents.append(lemmatized_sent)
    return doc_lemmatized_sents

In [13]:
work_ids = emlap_metadata["No."]
emlap_lemmatized_sents = []
for work_id in work_ids:
    emlap_lemmatized_sents.extend(extract_lemmatized_sents(work_id))

In [16]:
# total number of sentences:
len(emlap_lemmatized_sents)

220846